# Datos faltantes

En este ejercicio se identifican y rellenan los valores faltantes del conjunto de datos Diabetes.

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.impute import SimpleImputer, KNNImputer

In [ ]:
# Cargar el conjunto de datos
diabetes = load_diabetes(as_frame=True)
data = diabetes.data.copy()
data['target'] = diabetes.target

# Guardar una copia antes de simular los valores faltantes
data_original = data.copy()

In [ ]:
# Simular valores faltantes
np.random.seed(42)
indices_bmi = data.sample(frac=0.1, random_state=42).index
indices_bp = data.sample(frac=0.1, random_state=24).index
data.loc[indices_bmi, 'bmi'] = np.nan
data.loc[indices_bp, 'bp'] = np.nan

data.head()

## Ejercicios

### 1. ¿Cuántos valores faltantes hay en cada columna?

In [ ]:
# Contar los valores faltantes por columna
faltantes = data.isnull().sum()
print(faltantes)

### 2. Utiliza imputación simple (media) para llenar los valores faltantes de la columna `bmi`.

In [ ]:
# Reemplazar los valores faltantes de bmi por su media
imputador_media = SimpleImputer(strategy='mean')
data['bmi_media'] = imputador_media.fit_transform(data[['bmi']]).ravel()

print('Media utilizada:', round(data['bmi'].mean(), 4))
print('Valores faltantes después de imputar:', data['bmi_media'].isnull().sum())
data[['bmi', 'bmi_media']].head(10)

### 3. Utiliza KNNImputer para imputar valores en las columnas `bmi` y `bp`. Compara los resultados con los de la imputación simple.

In [ ]:
# Imputar bmi y bp usando los 5 vecinos más cercanos
imputador_knn = KNNImputer(n_neighbors=5)
valores_knn = imputador_knn.fit_transform(data[['bmi', 'bp']])
data['bmi_knn'] = valores_knn[:, 0]
data['bp_knn'] = valores_knn[:, 1]

comparacion = pd.DataFrame({
    'Método': ['Media', 'KNN'],
    'Promedio BMI': [data['bmi_media'].mean(), data['bmi_knn'].mean()],
    'Promedio BP': [data['bp'].mean(), data['bp_knn'].mean()]
})
print(comparacion.round(4).to_string(index=False))
print('\nValores imputados de ejemplo:')
print(data.loc[indices_bmi, ['bmi', 'bmi_media', 'bmi_knn']].head())

**Comparación:** La imputación por media asigna el mismo valor a todos los datos faltantes de `bmi`. KNN estima cada valor usando los registros más similares y puede conservar mejor las relaciones entre `bmi` y `bp`.

### 4. Genera un histograma comparando los datos antes y después de la imputación en la columna `bmi`.

In [ ]:
# Comparar la distribución original con los dos métodos de imputación
plt.figure(figsize=(10, 6))
sns.histplot(data_original['bmi'], color='gray', label='Datos originales', kde=True, stat='density', alpha=0.35)
sns.histplot(data['bmi_media'], color='royalblue', label='Imputación por media', kde=True, stat='density', alpha=0.35)
sns.histplot(data['bmi_knn'], color='darkorange', label='Imputación KNN', kde=True, stat='density', alpha=0.35)
plt.title('Distribución de BMI antes y después de la imputación')
plt.xlabel('BMI')
plt.ylabel('Densidad')
plt.legend()
plt.tight_layout()
plt.show()